# Data Loading (Local-First)

This notebook is a clean, fast local workflow for Task 1.
It avoids heavy model training and focuses on:
- chunked loading
- memory optimization
- parquet conversion
- EDA-ready outputs

In [1]:
# Imports and project path setup
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(project_root))

from src.utils.config import get_config, get_system_info
from src.data.memory_utils import compare_memory, optimize_dtypes
from src.data.preprocessing import (
    build_city_traffic_dataframe,
    save_city_parquet,
    load_city_parquet,
    get_top_square_id,
    get_area_series,
    report_missing_telecom_files,
    )

cfg = get_config()
get_system_info()

{'os': 'Linux 6.17.0-23-generic',
 'python': '3.12.3',
 'processor': 'x86_64',
 'machine': 'x86_64'}

In [2]:
# Paths
raw_dir = cfg.data_raw
parquet_path = cfg.data_processed / "city_traffic.parquet"
raw_files = sorted(raw_dir.glob("sms-call-internet-mi-*.txt"))
len(raw_files), raw_files[:2]

(3,
 [PosixPath('/home/nanen-miracle-mbanaade/ML Code/mobile-network-time-series-forecasting/data/raw/telecom/sms-call-internet-mi-2013-11-01.txt'),
  PosixPath('/home/nanen-miracle-mbanaade/ML Code/mobile-network-time-series-forecasting/data/raw/telecom/sms-call-internet-mi-2013-11-02.txt')])

In [3]:
# Check coverage of expected daily files (Nov 1 to Dec 31)
missing_report = report_missing_telecom_files(raw_dir)
missing_report, missing_report["missing_files"][:5]

({'expected': 61,
  'present': 3,
  'missing': 58,
  'missing_files': ['sms-call-internet-mi-2013-11-04.txt',
   'sms-call-internet-mi-2013-11-05.txt',
   'sms-call-internet-mi-2013-11-06.txt',
   'sms-call-internet-mi-2013-11-07.txt',
   'sms-call-internet-mi-2013-11-08.txt',
   'sms-call-internet-mi-2013-11-09.txt',
   'sms-call-internet-mi-2013-11-10.txt',
   'sms-call-internet-mi-2013-11-11.txt',
   'sms-call-internet-mi-2013-11-12.txt',
   'sms-call-internet-mi-2013-11-13.txt',
   'sms-call-internet-mi-2013-11-14.txt',
   'sms-call-internet-mi-2013-11-15.txt',
   'sms-call-internet-mi-2013-11-16.txt',
   'sms-call-internet-mi-2013-11-17.txt',
   'sms-call-internet-mi-2013-11-18.txt',
   'sms-call-internet-mi-2013-11-19.txt',
   'sms-call-internet-mi-2013-11-20.txt',
   'sms-call-internet-mi-2013-11-21.txt',
   'sms-call-internet-mi-2013-11-22.txt',
   'sms-call-internet-mi-2013-11-23.txt',
   'sms-call-internet-mi-2013-11-24.txt',
   'sms-call-internet-mi-2013-11-25.txt',
   'sms-

In [4]:
# Memory optimization probe on a small sample
sample_file = raw_files[0]
raw = pd.read_csv(
    sample_file,
    sep="\t",
    header=None,
    names=[
        "square_id",
        "time_interval",
        "country_code",
        "sms_in",
        "sms_out",
        "call_in",
        "call_out",
        "internet_traffic",
    ],
    usecols=[0, 1, 7],
    nrows=300_000,
 )
optimized = optimize_dtypes(raw)
cmp = compare_memory(raw, optimized)
{
    "before_mb": round(cmp.before_mb, 2),
    "after_mb": round(cmp.after_mb, 2),
    "reduction_pct": round(cmp.reduction_pct, 2),
}

{'before_mb': 6.87, 'after_mb': 4.01, 'reduction_pct': 41.67}

In [5]:
# Build parquet once (chunked). Skip if it already exists.
if not parquet_path.exists():
    city_df = build_city_traffic_dataframe(raw_dir, chunksize=2_000_000)
    save_city_parquet(city_df, parquet_path)
    print(f"Parquet saved to {parquet_path}")
else:
    print(f"Parquet already exists: {parquet_path}")

Parquet already exists: /home/nanen-miracle-mbanaade/ML Code/mobile-network-time-series-forecasting/data/processed/city_traffic.parquet


In [6]:
# Load parquet for fast EDA
city_df = load_city_parquet(parquet_path)
city_df.head()

,square_id,time_interval,internet_traffic
0,1,2013-10-31 23:00:00+00:00,11.028366
1,1,2013-10-31 23:10:00+00:00,11.127101
2,1,2013-10-31 23:20:00+00:00,10.892771
3,1,2013-10-31 23:30:00+00:00,8.622424
4,1,2013-10-31 23:40:00+00:00,8.009928


In [7]:
# Identify the highest-traffic area and check required squares
top_square = get_top_square_id(city_df)
top_square

5161

In [8]:
# Extract the three target series for Task 2
series_top = get_area_series(city_df, top_square)
series_4159 = get_area_series(city_df, 4159)
series_4556 = get_area_series(city_df, 4556)
series_top.head()

time_interval
2013-10-31 23:00:00+00:00    379.972931
2013-10-31 23:10:00+00:00    371.122711
2013-10-31 23:20:00+00:00    429.297943
2013-10-31 23:30:00+00:00    350.184235
2013-10-31 23:40:00+00:00    268.586823
Name: internet_traffic, dtype: float32

## Next (fast local)
Move to EDA notebook for plots, stationarity, decomposition, and spatial analysis.

## Later (GPU)
Move LSTM/GRU training and long tuning to Kaggle or Colab.